# Interactive 2024 error map with Leafmap

This notebook is the interactive spatial check. It follows the course visualisation guidance: use a basemap for context, a sequential scale for MAE, a diverging scale for signed bias, clear layer names, and keep optional diagnostic layers switched off by default.


## 1. Setup

In [1]:
# Run once if needed:
# %pip install -q leafmap

from pathlib import Path
import numpy as np
import pandas as pd
from pyproj import Transformer

import folium
from folium.plugins import Fullscreen, MiniMap, MeasureControl, HeatMap
from branca.colormap import LinearColormap
import leafmap.foliumap as leafmap

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

pred_file = PROJECT_ROOT / "data" / "processed" / "test_predictions_2024.parquet"
if not pred_file.exists():
    raise FileNotFoundError(
        "Run notebooks/01_main_project.ipynb first so the 2024 predictions exist."
    )

results = pd.read_parquet(pred_file)
print("Rows:", f"{len(results):,}")

Rows: 267,180


## 2. Prepare one row per sampled grid cell

In [2]:
E_DIM = next(c for c in results.columns if str(c).upper() == "E")
N_DIM = next(c for c in results.columns if str(c).upper() == "N")

cell_map = (
    results.groupby(
        ["cell_id", E_DIM, N_DIM, "elevation_m"],
        as_index=False,
    )
    .agg(
        MAE_C=("abs_error", "mean"),
        bias_C=("error", "mean"),
    )
)

transformer = Transformer.from_crs(
    "EPSG:2056",
    "EPSG:4326",
    always_xy=True,
)

lon, lat = transformer.transform(
    cell_map[E_DIM].to_numpy(),
    cell_map[N_DIM].to_numpy(),
)

cell_map["longitude"] = lon
cell_map["latitude"] = lat

cell_map.to_csv(
    PROJECT_ROOT / "results" / "cell_error_2024_wgs84.csv",
    index=False,
)

display(cell_map.head())

,cell_id,E,N,elevation_m,MAE_C,bias_C,longitude,latitude
0,0,2570500.0,1080500.0,1904.1,1.622272,-0.107738,7.058745,45.875475
1,1,2578500.0,1080500.0,2685.6,1.673725,-0.010006,7.161770,45.875778
2,2,2586500.0,1080500.0,2556.9,1.638308,-0.093461,7.264795,45.875985
3,3,2594500.0,1080500.0,3073.3,1.709939,-0.153630,7.367821,45.876098
4,4,2714500.0,1080500.0,450.6,1.125800,-0.035934,8.913040,45.866447


## 3. Build the interactive map

The heatmap is useful for the overall pattern. The marker layers are better for checking individual locations.

In [3]:
m = leafmap.Map(
    center=[46.8, 8.2],
    zoom=8,
    height="720px",
    tiles=None,
)

# Clean default context plus an optional terrain basemap.
folium.TileLayer(
    tiles="CartoDB positron",
    name="Clean basemap",
    control=True,
    show=True,
).add_to(m)

folium.TileLayer(
    tiles="https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png",
    attr="OpenTopoMap",
    name="Topographic basemap",
    control=True,
    show=False,
).add_to(m)

# Sequential MAE scale: low -> high error.
mae_min = float(cell_map["MAE_C"].min())
mae_max = float(cell_map["MAE_C"].max())
mae_scale = LinearColormap(
    ["#440154", "#3b528b", "#21918c", "#5ec962", "#fde725"],
    vmin=mae_min,
    vmax=mae_max,
    caption="2024 grid-cell MAE (°C)",
)

mae_group = folium.FeatureGroup(name="2024 MAE by grid cell", show=True)
for row in cell_map.itertuples(index=False):
    popup = folium.Popup(
        (
            f"<b>Grid cell {int(row.cell_id)}</b><br>"
            f"MAE: {row.MAE_C:.2f} °C<br>"
            f"Mean bias: {row.bias_C:+.2f} °C<br>"
            f"Elevation: {row.elevation_m:.0f} m"
        ),
        max_width=230,
    )
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=4.2,
        color=mae_scale(row.MAE_C),
        fill=True,
        fill_color=mae_scale(row.MAE_C),
        fill_opacity=0.82,
        weight=0.5,
        tooltip=f"MAE {row.MAE_C:.2f} °C | elevation {row.elevation_m:.0f} m",
        popup=popup,
    ).add_to(mae_group)
mae_group.add_to(m)
mae_scale.add_to(m)

# Smooth overview is useful for exploration but hidden by default because it
# visually spreads values beyond the actual sampled cells.
heat_group = folium.FeatureGroup(name="MAE heatmap (diagnostic)", show=False)
HeatMap(
    cell_map[["latitude", "longitude", "MAE_C"]].to_numpy().tolist(),
    radius=16,
    blur=14,
    min_opacity=0.15,
).add_to(heat_group)
heat_group.add_to(m)

# Signed bias: blue = underprediction, red = overprediction.
bias_limit = max(0.1, float(cell_map["bias_C"].abs().quantile(0.99)))
bias_scale = LinearColormap(
    ["#2166ac", "#f7f7f7", "#b2182b"],
    vmin=-bias_limit,
    vmax=bias_limit,
)

bias_group = folium.FeatureGroup(name="2024 mean bias (blue low / red high)", show=False)
for row in cell_map.itertuples(index=False):
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=4.2,
        color=bias_scale(row.bias_C),
        fill=True,
        fill_color=bias_scale(row.bias_C),
        fill_opacity=0.82,
        weight=0.5,
        tooltip=f"Bias {row.bias_C:+.2f} °C | MAE {row.MAE_C:.2f} °C",
        popup=folium.Popup(
            (
                f"<b>Grid cell {int(row.cell_id)}</b><br>"
                f"Mean bias: {row.bias_C:+.2f} °C<br>"
                f"MAE: {row.MAE_C:.2f} °C<br>"
                f"Elevation: {row.elevation_m:.0f} m"
            ),
            max_width=230,
        ),
    ).add_to(bias_group)
bias_group.add_to(m)

# Elevation layer helps inspect whether high-error cells coincide with terrain.
elev_min = float(cell_map["elevation_m"].min())
elev_max = float(cell_map["elevation_m"].max())
elev_scale = LinearColormap(
    ["#440154", "#31688e", "#35b779", "#fde725"],
    vmin=elev_min,
    vmax=elev_max,
)
elev_group = folium.FeatureGroup(name="Elevation by grid cell", show=False)
for row in cell_map.itertuples(index=False):
    folium.CircleMarker(
        location=[row.latitude, row.longitude],
        radius=4.0,
        color=elev_scale(row.elevation_m),
        fill=True,
        fill_color=elev_scale(row.elevation_m),
        fill_opacity=0.75,
        weight=0.4,
        tooltip=f"Elevation {row.elevation_m:.0f} m | MAE {row.MAE_C:.2f} °C",
    ).add_to(elev_group)
elev_group.add_to(m)

# Fit the map to the actual project extent.
m.fit_bounds([
    [float(cell_map["latitude"].min()), float(cell_map["longitude"].min())],
    [float(cell_map["latitude"].max()), float(cell_map["longitude"].max())],
])

# Small title/interpretation box rather than multiple permanent legends.
title_html = """
<div style="position: fixed; top: 12px; left: 50%; transform: translateX(-50%);
            z-index: 9999; background: rgba(255,255,255,0.92); padding: 8px 12px;
            border: 1px solid #bbb; border-radius: 5px; font-size: 14px;">
<b>2024 next-day temperature error across Switzerland</b><br>
<span style="font-size:12px">Default layer: MAE. Bias layer: blue = underprediction, red = overprediction.</span>
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

Fullscreen(position="topright").add_to(m)
MiniMap(toggle_display=True).add_to(m)
MeasureControl(position="topleft", primary_length_unit="kilometers").add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

m


## 4. Save a standalone HTML map

In [4]:
output = PROJECT_ROOT / "figures" / "leafmap_2024_error_enhanced.html"
m.to_html(str(output))

print("Saved: figures/leafmap_2024_error_enhanced.html")
print("If VS Code does not render the map inline, open this HTML file in a normal browser.")


Saved: figures/leafmap_2024_error_enhanced.html
If VS Code does not render the map inline, open this HTML file in a normal browser.


## What to look for

Use the map for spatial inspection, not as a replacement for the numeric metrics.

- MAE layer: where is the model consistently less accurate?
- Bias layer: are there coherent regions that are systematically too warm or too cold?
- Elevation layer: do difficult cells line up with higher terrain?
- Topographic basemap: does the error pattern follow Alpine terrain or valleys?

For the final report, use one focused static map plus a short caption. Keep this interactive map as an exploratory / repository output.
